In [5]:

#import the required libraries
import pandas as pd
import numpy as np
import rasterio
import json
import os
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
from sklearn.neighbors import BallTree

In [6]:
# Load Surface Features
surface = pd.read_csv('../data/processed/surface_features_clean.csv')
 
print("Surface features loaded")
print("Shape:", surface.shape)
print("Columns:", surface.columns.tolist())
print("Nulls:", surface.isnull().sum().sum())

print(surface.head(3))

Surface features loaded
Shape: (4524, 8)
Columns: ['lat', 'lon', 'elevation', 'slope', 'aspect', 'curvature', 'NDVI', 'rainfall']
Nulls: 0
         lat        lon  elevation      slope      aspect  curvature  \
0  30.692876  78.715304       2641  15.275303  310.692688       17.0   
1  30.782188  79.305341       4942  41.546707  267.906311       11.0   
2  30.384875  78.731183       1372  21.407080  253.199554        7.0   

       NDVI     rainfall  
0  0.809694  1480.279615  
1  0.063395  1378.535364  
2  0.698203  1389.197917  


In [7]:
# Verify Competency Map
COMPETENCY_PATH = '../T4 1/Data/Outputs/competency_map.tif'
 
with rasterio.open(COMPETENCY_PATH) as src:
    crs  = src.crs
    res  = src.res
    bounds = src.bounds
    shape  = src.shape
    nodata = src.nodata
    data_check = src.read(1)
 
    # Mask nodata
    if nodata is not None:
        valid = data_check[data_check != nodata]
    else:
        valid = data_check.flatten()
 
    val_min  = valid.min()
    val_max  = valid.max()
    coverage = (len(valid) / data_check.size) * 100

crs_ok = str(crs) == 'EPSG:4326'
res_ok = abs(res[0] - 0.000277) < 0.00005
bounds_ok = (abs(bounds.left - 78.2) < 0.05 and
             abs(bounds.bottom - 30.0) < 0.05 and
             abs(bounds.right - 80.0) < 0.05 and
             abs(bounds.top - 31.0) < 0.05)
range_ok  = val_min >= 0.0 and val_max <= 1.0
nodata_ok = nodata is not None
coverage_ok = coverage > 85.0
print(f"CRS  : {crs} {'(PASS)' if crs_ok     else 'FAIL — must be EPSG:4326'}")
print(f"Resolution : {res} {'(PASS)' if res_ok     else 'FAIL — must be ~0.000277'}")
print(f"Bounds : {bounds} {'(PASS)' if bounds_ok  else 'FAIL — must cover 78.2-80.0, 30.0-31.0'}")
print(f"Value range : {val_min:.3f} to {val_max:.3f}  {'(PASS)' if range_ok   else 'FAIL — must be 0 to 1'}")
print(f"Nodata : {nodata}  -->  {'PASS' if nodata_ok  else 'FAIL — nodata must be defined'}")
print(f"Coverage : {coverage:.1f}%   {'(PASS)' if coverage_ok else 'FAIL — must be above 85%'}")

all_pass = crs_ok and res_ok and bounds_ok and range_ok and nodata_ok and coverage_ok
if all_pass:
    print("ALL CHECKS PASSED")
else:
    print("SOME CHECKS FAILED")
    raise ValueError("Competency map failed verification. Fix before proceeding.")

CRS  : EPSG:4326 (PASS)
Resolution : (0.00027700831024930707, 0.0002770083102493075) (PASS)
Bounds : BoundingBox(left=78.2, bottom=30.0, right=80.0, top=31.0) (PASS)
Value range : 0.100 to 0.680  (PASS)
Nodata : -9999.0  -->  PASS
Coverage : 100.0%   (PASS)
ALL CHECKS PASSED


In [10]:
# sample Competency Values at Surface Feature Points
with rasterio.open(COMPETENCY_PATH) as src:
    raster_data = src.read(1)
    nodata_val = src.nodata
    competency_values = []
 
    for idx, row in surface.iterrows():
        try:
            row_idx, col_idx = src.index(row['lon'], row['lat'])
 
            # Check bounds
            if (0 <= row_idx < raster_data.shape[0] and
                0 <= col_idx < raster_data.shape[1]):
 
                value = raster_data[row_idx, col_idx]
 
                # Check for nodata
                if nodata_val is not None and value == nodata_val:
                    competency_values.append(np.nan)
                else:
                    competency_values.append(float(value))
            else:
                # Point falls outside raster bounds
                competency_values.append(np.nan)
 
        except Exception as e:
            competency_values.append(np.nan)

surface['competency_index'] = competency_values
print("Null competency values  :", surface['competency_index'].isna().sum())
print("Valid competency values :", surface['competency_index'].notna().sum())
print("Competency range :", surface['competency_index'].min().round(3),"to",surface['competency_index'].max().round(3))

# Drop points where competency could not be sampled
before = len(surface)
surface = surface.dropna(subset=['competency_index'])
after  = len(surface)
print(f"Rows dropped (outside raster) : {before - after}")
print(f"Rows remaining  : {after}")

# If more than 20% of rows were dropped, something is wrong
drop_pct = ((before - after) / before) * 100
if drop_pct > 20:
    print(f"WARNING: {drop_pct:.1f}% of surface points returned null competency.")
    
    raise ValueError("Too many null competency values — competency map coverage insufficient.")
else:
    print(f"Acceptable null rate: {drop_pct:.1f}%")

Null competency values  : 0
Valid competency values : 4524
Competency range : 0.1 to 0.68
Rows dropped (outside raster) : 0
Rows remaining  : 4524
Acceptable null rate: 0.0%


In [11]:
# Calculate Factor of Safety (Physics Feature)
 
# The Infinite Slope Equation
# FS = [c + (gamma - gamma_w * m) * z * cos(B)^2 * tan(phi)] / [gamma * z * sin(B) * cos(B)]
#
# Where:
#   c  = cohesion from competency (competency * 20 kPa)
#   gamma = unit weight of material (20 kN/m3)
#   gamma_w = unit weight of water (9.81 kN/m3)
#   m = pore pressure ratio (rainfall normalized 0-1)
#   z = assumed regolith depth (2.0 m)
#   B  = slope angle in radians
#   phi = friction angle (30 + competency*10 degrees)

def factor_of_safety(row):
    c  = row['competency_index'] * 20.0
    gamma  = 20.0
    gamma_w = 9.81
    m  = min(row['rainfall'] / 3000.0, 1.0)
    z  = 2.0
    beta = np.radians(row['slope'])
    phi = np.radians(30.0 + row['competency_index'] * 10.0)
 
    sin_b = np.sin(beta)
    cos_b = np.cos(beta)
 
    # Avoid division by zero for flat terrain
    denominator = gamma * z * sin_b * cos_b
    if abs(denominator) < 1e-9:
        return 999.0  # flat ground is infinitely stable
 
    numerator = (c + (gamma - gamma_w * m) * z * (cos_b ** 2) * np.tan(phi))
 
    fs = numerator / denominator
 
    # Cap FS at 999 for very stable slopes and floor at 0
    return max(0.0, min(fs, 999.0))

surface['FS'] = surface.apply(factor_of_safety, axis=1)
 
# Cap extreme FS values for ML purposes
# Very high FS values (>10) all mean "very stable" — no need for distinction
surface['FS'] = surface['FS'].clip(upper=10.0)

print("FS range :", surface['FS'].min().round(3),"to", surface['FS'].max().round(3))
print("FS mean :", surface['FS'].mean().round(3))


# Sanity check — steep slopes with high rainfall should have low FS
steep_wet = surface[(surface['slope'] > 40) & (surface['rainfall'] > 1500)]
print("Mean FS for steep high-rainfall areas (should be < 2.0):", steep_wet['FS'].mean().round(3))
 
flat_dry = surface[(surface['slope'] < 10) & (surface['rainfall'] < 800)]
print("Mean FS for flat dry areas (should be > 5.0):",flat_dry['FS'].mean().round(3))

FS range : 0.489 to 10.0
FS mean : 1.964
Mean FS for steep high-rainfall areas (should be < 2.0): 0.953
Mean FS for flat dry areas (should be > 5.0): 7.135


In [13]:
# Load Landslide Labels

labels = pd.read_csv('../T4 1/Data/Processed/landslide_labels.csv')
 
print("Shape:", labels.shape)
print("Columns :", labels.columns.tolist())

print(labels['label'].value_counts())

 
# Verify expected columns exist
required_cols = ['lat', 'lon', 'label']
for col in required_cols:
    if col not in labels.columns:
        raise ValueError(f"Missing required column '{col}' in landslide_labels.csv")



# Verify labels are only 0 and 1
unique_labels = labels['label'].unique()
print("Unique label values:", unique_labels)
if not all(l in [0, 1] for l in unique_labels):
    raise ValueError("Labels must be 0 or 1 only")
 
# Verify label points are within study area
out_of_bounds = labels[
    (labels['lon'] < 78.2) | (labels['lon'] > 80.0) |
    (labels['lat'] < 30.0) | (labels['lat'] > 31.0)
]
if len(out_of_bounds) > 0:
    print(f"WARNING: {len(out_of_bounds)} label points are outside the study area bounding box")
    print("Removing out-of-bounds points...")
    labels = labels[
        (labels['lon'] >= 78.2) & (labels['lon'] <= 80.0) &
        (labels['lat'] >= 30.0) & (labels['lat'] <= 31.0)
    ]
    print("Remaining label points:", len(labels))        

Shape: (242, 4)
Columns : ['point_id', 'lat', 'lon', 'label']
label
1    121
0    121
Name: count, dtype: int64
Unique label values: [1 0]


In [15]:
# Match Label Points to Nearest Surface Feature Point

# We use BallTree spatial nearest-neighbor matching
# Each label point gets assigned the feature values of the nearest surface point from the GEE extraction
 
 
# Build spatial index on surface points (in radians for haversine)
surface_coords = np.radians(surface[['lat', 'lon']].values)
label_coords = np.radians(labels[['lat', 'lon']].values)
 
tree = BallTree(surface_coords, metric='haversine')
 
# Find nearest surface point for each label point
# k=1 means find the single nearest neighbor
distances, indices = tree.query(label_coords, k=1)
 
# Convert distances from radians to kilometres
distances_km = distances.flatten() * 6371

print("Distance stats (km from label to nearest surface point):")
print(f" Min : {distances_km.min():.2f} km")
print(f" Max : {distances_km.max():.2f} km")
print(f" Mean: {distances_km.mean():.2f} km")


# Flag any label points that are very far from any surface point
# More than 2km away suggests a coverage gap
far_points = (distances_km > 2.0).sum()
if far_points > 0:
    print(f"WARNING: {far_points} label points are more than 2km from nearest surface feature.")

Distance stats (km from label to nearest surface point):
 Min : 0.04 km
 Max : 3.18 km
 Mean: 1.07 km


In [16]:
#Build Master Feature Table


feature_cols = ['elevation', 'slope', 'aspect', 'curvature','NDVI', 'rainfall', 'competency_index', 'FS']
 
# Get feature values for each label point from its nearest surface point
matched_features = surface.iloc[indices.flatten()][feature_cols].reset_index(drop=True)
 
# Combine with labels
labels_reset = labels.reset_index(drop=True)
 
master = pd.concat([labels_reset, matched_features], axis=1)
 
# Add distance column for reference
master['nearest_dist_km'] = distances_km

print("Shape :", master.shape)
print("Columns :", master.columns.tolist())

print("Label distribution:")
print(master['label'].value_counts())

print("Null check:")
print(master[feature_cols].isnull().sum())

print("Feature statistics:")
print(master[feature_cols].describe().round(3))

Shape : (242, 13)
Columns : ['point_id', 'lat', 'lon', 'label', 'elevation', 'slope', 'aspect', 'curvature', 'NDVI', 'rainfall', 'competency_index', 'FS', 'nearest_dist_km']
Label distribution:
label
1    121
0    121
Name: count, dtype: int64
Null check:
elevation           0
slope               0
aspect              0
curvature           0
NDVI                0
rainfall            0
competency_index    0
FS                  0
dtype: int64
Feature statistics:
       elevation    slope   aspect  curvature     NDVI  rainfall  \
count    242.000  242.000  242.000    242.000  242.000   242.000   
mean    2380.711   31.750  175.781     -0.219    0.527  1311.077   
std     1352.312   12.285  104.764     38.349    0.251   263.928   
min      434.000    1.081   -0.000   -163.000   -0.371   494.036   
25%     1298.000   23.671   82.150    -20.750    0.403  1144.302   
50%     1974.000   32.095  184.344     -1.000    0.568  1286.292   
75%     3237.500   39.405  261.498     17.000    0.726  148

In [18]:
 # Final Data Quality Checks


# Check 1 — No nulls in features
null_count = master[feature_cols].isnull().sum().sum()


# Check 2 — Label balance
label_counts = master['label'].value_counts()
n_pos = label_counts.get(1, 0)
n_neg = label_counts.get(0, 0)
ratio = min(n_pos, n_neg) / max(n_pos, n_neg) if max(n_pos, n_neg) > 0 else 0
balance_ok = ratio > 0.6  # allow up to 40/60 split
print(f"Label balance (0:{n_neg}, 1:{n_pos}, ratio:{ratio:.2f}){'(PASS)' if balance_ok else 'WARNING — consider rebalancing'}")
 
# Check 3 — NDVI range
ndvi_ok = master['NDVI'].min() < 0.2 and master['NDVI'].max() > 0.5
print(f"NDVI range ({master['NDVI'].min():.3f} to {master['NDVI'].max():.3f}){'(PASS)' if ndvi_ok else 'WARNING — narrow range'}")
 
# Check 4 — Slope range
slope_ok = master['slope'].max() > 30
print(f"Slope range ({master['slope'].min():.1f} to {master['slope'].max():.1f} deg){'(PASS)' if slope_ok else 'WARNING — no steep slopes found'}")


# Check 5 — Competency range
comp_ok = master['competency_index'].min() >= 0 and master['competency_index'].max() <= 1
print(f"Competency range ({master['competency_index'].min():.3f} to {master['competency_index'].max():.3f}){'(PASS)' if comp_ok else 'FAIL'}")
 
# Check 6 — FS range
fs_ok = master['FS'].min() >= 0
print(f"FS range ({master['FS'].min():.3f} to {master['FS'].max():.3f}){'(PASS)' if fs_ok else 'FAIL'}")
 
# Check 7 — Minimum sample size
size_ok = len(master) >= 100
print(f"Total samples ({len(master)}) {'(PASS)' if size_ok else 'FAIL — too few samples for training'}")



Label balance (0:121, 1:121, ratio:1.00)(PASS)
NDVI range (-0.371 to 0.980)(PASS)
Slope range (1.1 to 68.5 deg)(PASS)
Competency range (0.100 to 0.680)(PASS)
FS range (0.727 to 10.000)(PASS)
Total samples (242) (PASS)


In [21]:
# Save Master Features CSV
# Drop the distance column before saving as it is not a model feature
master_save = master.drop(columns=['nearest_dist_km'])
 
os.makedirs('data/processed', exist_ok=True)
master_save.to_csv('../data/processed/master_features.csv',index=False)